In [1]:
import sys
print(sys.executable)
print(sys.version)

d:\Ramanasoft\DATA_SCIENCE_INTERNS\Projects\Real_Estate_Investment_Advisor\.venv314\Scripts\python.exe
3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("All libraries imported successfully!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

All libraries imported successfully!
Pandas: 3.1.1
NumPy: 2.5.4
Scikit-learn: 1.9.1


In [3]:
import pandas as pd

df = pd.read_csv("../data/india_housing_prices.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset Shape: (250000, 23)


,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,...,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,...,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,...,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,300.29,0.11,1991,...,34,5,7,High,Yes,Yes,"Playground, Clubhouse, Gym, Pool, Garden",North,Builder,Ready_to_Move
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,182.90,0.04,2002,...,23,4,9,Low,No,Yes,"Playground, Garden, Gym, Pool, Clubhouse",East,Builder,Ready_to_Move


# Load and understand the dataset

#### Import libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


##### Load the dataset

In [5]:
df = pd.read_csv("../data/india_housing_prices.csv")

print("Dataset loaded successfully!")

Dataset loaded successfully!


# Step 1: Data Preprocessing

##### ●	Handle missing values and duplicates.

In [6]:
df_clean = df.copy()
print("Shape before:", df_clean.shape)

# Treat blanks / 'NA' / 'null' etc. in text columns as missing
text_cols = df_clean.select_dtypes(exclude="number").columns
df_clean[text_cols] = (df_clean[text_cols]
                       .apply(lambda s: s.str.strip())
                       .replace(["", "NA", "N/A", "na", "null", "None", "nan", "?"], np.nan))
print("Missing cells:", df_clean.isna().sum().sum())

# Fill: median for numeric columns, mode for text columns
num_cols = df_clean.select_dtypes(include="number").columns.drop("ID")
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())
df_clean[text_cols] = df_clean[text_cols].fillna(df_clean[text_cols].mode().iloc[0])

# Drop duplicates (ID excluded because it is unique per row)
df_clean = df_clean.drop_duplicates(subset=df_clean.columns.drop("ID")).reset_index(drop=True)
print("Missing after:", df_clean.isna().sum().sum(), "| Shape after:", df_clean.shape)

Shape before: (250000, 23)
Missing cells: 0
Missing after: 0 | Shape after: (250000, 23)


##### ●	Normalize or scale numerical features.

In [7]:
from sklearn.preprocessing import StandardScaler

# Numeric columns (ID is just an identifier, so it's excluded)
num_cols = df_clean.select_dtypes(include="number").columns.drop("ID").tolist()

# Scale into a separate copy so df_clean keeps the original values
df_scaled = df_clean.copy()
scaler = StandardScaler()
df_scaled[num_cols] = scaler.fit_transform(df_clean[num_cols])

print(df_scaled[num_cols].agg(["mean", "std"]).round(3).T)

                  mean  std
BHK                0.0  1.0
Size_in_SqFt      -0.0  1.0
Price_in_Lakhs     0.0  1.0
Price_per_SqFt    -0.0  1.0
Year_Built         0.0  1.0
Floor_No           0.0  1.0
Total_Floors      -0.0  1.0
Age_of_Property    0.0  1.0
Nearby_Schools     0.0  1.0
Nearby_Hospitals   0.0  1.0


##### ●	Encode categorical features like Location and Property_Type.

In [8]:
df_enc = df_clean.copy()

# 1. Binary Yes/No -> 0/1
for c in ["Parking_Space", "Security"]:
    df_enc[c] = df_enc[c].map({"No": 0, "Yes": 1})

# 2. Ordinal (has a natural order) -> 0/1/2
df_enc["Furnished_Status"] = df_enc["Furnished_Status"].map(
    {"Unfurnished": 0, "Semi-furnished": 1, "Furnished": 2})
df_enc["Public_Transport_Accessibility"] = df_enc["Public_Transport_Accessibility"].map(
    {"Low": 0, "Medium": 1, "High": 2})

# 3. Amenities (comma-separated text) -> one flag per amenity + a count
amen = df_enc["Amenities"].str.get_dummies(sep=", ").add_prefix("Amenity_")
df_enc = pd.concat([df_enc.drop(columns="Amenities"), amen], axis=1)
df_enc["Amenities_Count"] = amen.sum(axis=1)

# 4. Locality (500 values) -> frequency encoding
df_enc["Locality_Freq"] = df_enc["Locality"].map(df_enc["Locality"].value_counts(normalize=True))
df_enc = df_enc.drop(columns="Locality")

# 5. Nominal, low-cardinality -> one-hot
ohe_cols = ["Property_Type", "Facing", "Owner_Type", "Availability_Status", "State", "City"]
df_enc = pd.get_dummies(df_enc, columns=ohe_cols, drop_first=True, dtype=int)

print(df_enc.shape)
print(df_enc.select_dtypes(exclude="number").columns.tolist())   # should be []

(250000, 90)
[]


##### ●	Create new features like Price per Sqft, School Density Score, etc.

In [9]:
df_fe = df_clean.copy()

# 1. Price per sq ft in rupees (the original column is in lakhs)
df_fe["Price_per_SqFt_Rs"] = df_fe["Price_in_Lakhs"] * 100000 / df_fe["Size_in_SqFt"]

# 2. Average room size
df_fe["Size_per_BHK"] = df_fe["Size_in_SqFt"] / df_fe["BHK"]

# 3. School and hospital density scores (scaled 0 to 1)
minmax = lambda s: (s - s.min()) / (s.max() - s.min())
df_fe["School_Density_Score"] = minmax(df_fe["Nearby_Schools"])
df_fe["Hospital_Density_Score"] = minmax(df_fe["Nearby_Hospitals"])

# 4. Overall infrastructure score (schools + hospitals + transport, 0 to 1)
transport = df_fe["Public_Transport_Accessibility"].map({"Low": 0, "Medium": 0.5, "High": 1})
df_fe["Infrastructure_Score"] = (df_fe["School_Density_Score"]
                                 + df_fe["Hospital_Density_Score"] + transport) / 3

# 5. Number of amenities
df_fe["Amenities_Count"] = df_fe["Amenities"].str.count(",") + 1

# 6. How expensive vs the city norm
df_fe["City_Median_Price_per_SqFt"] = df_fe.groupby("City")["Price_per_SqFt_Rs"].transform("median")
df_fe["Price_vs_City_Median"] = df_fe["Price_per_SqFt_Rs"] / df_fe["City_Median_Price_per_SqFt"]

# 7. Property age group
df_fe["Age_Group"] = pd.cut(df_fe["Age_of_Property"], bins=[-1, 5, 15, 30, 100],
                            labels=["New", "Recent", "Mature", "Old"])
new_cols = ["Price_per_SqFt_Rs", "Size_per_BHK", "School_Density_Score", "Hospital_Density_Score",
            "Infrastructure_Score", "Amenities_Count", "City_Median_Price_per_SqFt",
            "Price_vs_City_Median", "Age_Group"]
print(df_fe.shape)
print(df_fe[new_cols].head().round(2))
print(df_fe[new_cols].describe().round(2).T[["min", "mean", "max"]])

(250000, 32)
   Price_per_SqFt_Rs  Size_per_BHK  School_Density_Score  \
0           10332.49       4740.00                  1.00   
1            8270.73        788.00                  0.78   
2            5046.40       1821.00                  0.89   
3           10955.49       1370.50                  0.44   
4            3792.25       1205.75                  0.33   

   Hospital_Density_Score  Infrastructure_Score  Amenities_Count  \
0                    0.22                  0.74                5   
1                    0.00                  0.26                5   
2                    0.78                  0.56                4   
3                    0.67                  0.70                5   
4                    0.89                  0.41                5   

   City_Median_Price_per_SqFt  Price_vs_City_Median Age_Group  
0                     9328.56                  1.11       Old  
1                     9538.74                  0.87    Mature  
2                     915

##### ●	Create a binary label "Good Investment" based on domain rules (e.g., appreciation rate > threshold).

In [10]:
# One point per rule (uses the columns created in the feature step)
df_fe["Rule_Value"]   = (df_fe["Price_per_SqFt_Rs"] <= df_fe["City_Median_Price_per_SqFt"]).astype(int)  # priced at or below the city norm
df_fe["Rule_Infra"]   = (df_fe["Infrastructure_Score"] >= 0.5).astype(int)                                # good schools/hospitals/transport
df_fe["Rule_Amenity"] = (df_fe["Amenities_Count"] >= 3).astype(int)                                       # well-equipped
df_fe["Rule_Age"]     = (df_fe["Age_of_Property"] <= 15).astype(int)                                      # relatively new
df_fe["Rule_Ready"]   = (df_fe["Availability_Status"] == "Ready_to_Move").astype(int)                     # can be used or rented now

rules = ["Rule_Value", "Rule_Infra", "Rule_Amenity", "Rule_Age", "Rule_Ready"]
df_fe["Investment_Score"] = df_fe[rules].sum(axis=1)

# Good investment = meets at least 3 of the 5 rules
df_fe["Good_Investment"] = (df_fe["Investment_Score"] >= 3).astype(int)

print(df_fe["Good_Investment"].value_counts())
print(df_fe["Good_Investment"].value_counts(normalize=True).round(3))

Good_Investment
1    127950
0    122050
Name: count, dtype: int64
Good_Investment
1    0.512
0    0.488
Name: proportion, dtype: float64


In [11]:
import os
os.makedirs("../data/processed", exist_ok=True)
df_fe.to_csv("../data/processed/housing_features.csv", index=False)

#### Future_Price_5Y

In [12]:
df_fe.to_csv("../data/processed/housing_features.csv", index=False)